In [ ]:
%run ./base_model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

In [ ]:
class XGBoostModel(BaseForecastModel):
    INT_PARAMS = ("max_depth", "n_estimators", "min_child_weight")
    DEFAULT_SPACE = {"max_depth": [3, 10], "n_estimators": [100, 300], "learning_rate": [0.01, 0.3]}

    def __init__(self, estimator):
        self.estimator = estimator

    @classmethod
    def build(cls, params):
        # Deferred xgboost import
        from xgboost import XGBRegressor
        p = cls._cast_int_params(params)
        return cls(XGBRegressor(random_state=0, n_jobs=-1, **p))

In [ ]:
class HistGradientBoostingModel(BaseForecastModel):
    INT_PARAMS = ("max_depth", "max_iter", "max_leaf_nodes", "min_samples_leaf")
    DEFAULT_SPACE = {"max_iter": [50, 200], "max_depth": [2, 6]}

    def __init__(self, estimator):
        self.estimator = estimator

    @classmethod
    def build(cls, params):
        p = cls._cast_int_params(params)
        return cls(HistGradientBoostingRegressor(random_state=0, **p))

In [ ]:
class DecisionTreeModel(BaseForecastModel):
    INT_PARAMS = ("max_depth",)
    DEFAULT_SPACE = {"max_depth": [2, 12]}

    def __init__(self, estimator):
        self.estimator = estimator

    @classmethod
    def build(cls, params):
        p = cls._cast_int_params(params)
        return cls(DecisionTreeRegressor(random_state=0, min_samples_leaf=5, **p))

In [ ]:
class LightGBMModel(BaseForecastModel):
    INT_PARAMS = ("max_depth", "n_estimators", "num_leaves", "min_child_samples")
    DEFAULT_SPACE = {"n_estimators": [50, 200], "max_depth": [2, 6], "num_leaves": [15, 63]}

    def __init__(self, estimator):
        self.estimator = estimator

    @classmethod
    def build(cls, params):
        # Deferred lightgbm import
        from lightgbm import LGBMRegressor
        p = cls._cast_int_params(params)
        # subsample_freq is required or LightGBM silently ignores subsample
        return cls(LGBMRegressor(random_state=0, n_jobs=-1, verbose=-1, subsample_freq=1, **p))

In [ ]:
class SarimaxRegressor(BaseEstimator):
    # sklearn-shaped wrapper: SARIMAX forecasts len(exog) steps ahead of
    # wherever the fitted series ends, so successive single-row predict()
    # calls (the recursive forecast loop) accumulate a future-exog buffer
    # and each call re-forecasts up to the buffer's new length, returning
    # only the newly requested rows.
    def __init__(self, p=1, d=1, q=1, P=0, D=0, Q=0, s=7):
        self.p, self.d, self.q = p, d, q
        self.P, self.D, self.Q, self.s = P, D, Q, s

    def _prepare(self, X):
        # SARIMAX rejects NaN exog and goes singular on constant columns
        X = pd.DataFrame(X)[self._exog_cols].astype(float).reset_index(drop=True)
        return X.fillna(self._fill).fillna(0.0)

    def fit(self, X, y):
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        y = pd.Series(y).astype(float).reset_index(drop=True)
        X = pd.DataFrame(X).astype(float).reset_index(drop=True).ffill().bfill()
        keep = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
        self._exog_cols = keep or list(X.columns)
        self._fill = X[self._exog_cols].median()
        exog = self._prepare(X)

        self.model_ = SARIMAX(
            y, exog=exog,
            order=(self.p, self.d, self.q),
            seasonal_order=(self.P, self.D, self.Q, self.s),
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        self._future_exog = None
        return self

    def reset_forecast(self):
        self._future_exog = None

    def advance(self, X_new, y_new):
        # Extend the fitted state with real observations, no refit
        self.model_ = self.model_.append(
            np.asarray(pd.Series(y_new).astype(float)),
            exog=np.asarray(self._prepare(X_new)),
            refit=False,
        )
        self._future_exog = None

    def predict(self, X):
        X = self._prepare(X)
        self._future_exog = X if self._future_exog is None else pd.concat(
            [self._future_exog, X], ignore_index=True
        )
        forecast = self.model_.get_forecast(
            steps=len(self._future_exog), exog=self._future_exog
        )
        return forecast.predicted_mean.values[-len(X):]


class SarimaxModel(BaseForecastModel):
    # Small grid on purpose: SARIMAX refits from scratch on every
    # search/CV fold, unlike the tree models it's slow per fit
    INT_PARAMS = ("p", "d", "q", "P", "D", "Q", "s")
    DEFAULT_SPACE = {"p": [0, 2], "q": [0, 2]}

    def __init__(self, estimator):
        self.estimator = estimator

    def reset_forecast(self):
        self.estimator.reset_forecast()

    def advance(self, X_new, y_new):
        self.estimator.advance(X_new, y_new)

    @classmethod
    def build(cls, params):
        # Deferred statsmodels import
        from statsmodels.tsa.statespace.sarimax import SARIMAX  # noqa: F401
        p = cls._cast_int_params(params)
        return cls(SarimaxRegressor(**p))

In [ ]:
# Model type registry
MODEL_REGISTRY = {
    "xgboost": XGBoostModel,
    "hist_gradient_boosting": HistGradientBoostingModel,
    "decision_tree": DecisionTreeModel,
    "lightgbm": LightGBMModel,
    "sarimax": SarimaxModel,
}